<a href="https://colab.research.google.com/github/beatrizzzzz/Estudo-com-o-PERMA-Profiler/blob/main/analise_mca_participantes_conceitos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import re
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

CSV_NAME = "respostas_felicidade_enriquecidas_CORRIGIDO_respostas_em_branco.csv"
IMAGE_OUT = "mca_participantes_espaco_semantico.png"


def localizar_ou_enviar_csv(nome_arquivo):
    """
    Procura o CSV em caminhos comuns.

    No Google Colab, caso não encontre o arquivo, abre a janela de upload.
    """
    caminhos_possiveis = [
        Path(nome_arquivo),
        Path("/content") / nome_arquivo,
        Path("/mnt/data") / nome_arquivo,
    ]

    for caminho in caminhos_possiveis:
        if caminho.exists():
            print(f"CSV encontrado em: {caminho}")
            return str(caminho)

    try:
        from google.colab import files

        print(
            "O CSV não foi encontrado automaticamente. "
            "Selecione o arquivo na janela de upload."
        )
        enviados = files.upload()

        if not enviados:
            raise FileNotFoundError("Nenhum arquivo foi enviado.")

        if nome_arquivo in enviados:
            caminho = Path(nome_arquivo)
        else:
            primeiro_arquivo = next(iter(enviados))
            caminho = Path(primeiro_arquivo)
            print(f"Arquivo selecionado: {primeiro_arquivo}")

        return str(caminho)

    except ImportError as exc:
        raise FileNotFoundError(
            f"O arquivo '{nome_arquivo}' não foi encontrado. "
            "Coloque o CSV na mesma pasta do notebook ou altere CSV_NAME."
        ) from exc


INPUT_CSV = localizar_ou_enviar_csv(CSV_NAME)


CSV encontrado em: respostas_felicidade_enriquecidas_CORRIGIDO_respostas_em_branco.csv


## 1. Funções auxiliares

In [ ]:
def clean_col(value):
    return (
        str(value)
        .replace("\ufeff", "")
        .replace("ï»¿", "")
        .strip()
    )


def parse_num(value):
    if pd.isna(value):
        return np.nan

    if isinstance(value, (int, float, np.number)):
        return float(value)

    text = str(value).strip().replace(",", ".")

    if not text:
        return np.nan

    # Corrige valores exportados como 3.333.333.333.
    parts = text.split(".")
    if len(parts) > 2 and all(part.isdigit() for part in parts):
        text = parts[0] + "." + "".join(parts[1:])

    try:
        return float(text)
    except ValueError:
        return np.nan


def norm_text(value):
    if pd.isna(value):
        return ""

    text = str(value).lower().strip()
    text = unicodedata.normalize("NFD", text)
    text = "".join(
        char for char in text
        if unicodedata.category(char) != "Mn"
    )
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def hit(text, patterns):
    return int(any(re.search(pattern, text) for pattern in patterns))


def tercile(series):
    numeric = pd.to_numeric(series, errors="coerce")
    ranks = numeric.rank(method="average", na_option="keep")

    try:
        return pd.qcut(
            ranks,
            3,
            labels=["Baixo", "Intermediario", "Alto"],
        ).astype(object)
    except ValueError:
        return pd.cut(
            numeric,
            3,
            labels=["Baixo", "Intermediario", "Alto"],
            include_lowest=True,
        ).astype(object)


## 2. Leitura e preparação dos dados

In [ ]:
df = pd.read_csv(INPUT_CSV, encoding="utf-8-sig")
df.columns = [clean_col(column) for column in df.columns]

if "ID" not in df.columns:
    candidates = [
        column
        for column in df.columns
        if column.endswith("ID") or "ID" in column
    ]

    if candidates:
        df = df.rename(columns={candidates[0]: "ID"})

if "Resposta_Felicidade" not in df.columns:
    raise ValueError(
        "A coluna 'Resposta_Felicidade' não foi encontrada. "
        f"Colunas disponíveis: {df.columns.tolist()}"
    )

numeric_columns = [
    "Escore_Geral",
    "Escore_P",
    "Escore_E",
    "Escore_R",
    "Escore_M",
    "Escore_A",
    "Idade",
    "Genero",
    "Escolaridade",
]

for column in numeric_columns:
    if column in df.columns:
        df[column] = df[column].apply(parse_num)

df["Resposta_Felicidade"] = (
    df["Resposta_Felicidade"]
    .fillna("")
    .astype(str)
    .str.strip()
)

base = (
    df.loc[df["Resposta_Felicidade"].ne("")]
    .copy()
    .reset_index(drop=True)
)

base["texto_normalizado"] = (
    base["Resposta_Felicidade"]
    .apply(norm_text)
)

print(f"Arquivo utilizado: {INPUT_CSV}")
print(f"Respostas utilizadas: {len(base)}")


Arquivo utilizado: respostas_felicidade_enriquecidas_CORRIGIDO_respostas_em_branco.csv
Respostas utilizadas: 87


## 3. Identificação dos conceitos nas respostas

In [ ]:
concepts = {
    "Relações afetivas": [
        r"\bfamili", r"\bfilh", r"\bmae\b", r"\bpai\b",
        r"\bavo", r"\bnet", r"\bamig", r"\bamor", r"\bamar",
        r"\bamado", r"\bcompanh", r"\brelacion",
        r"\bpessoas? que amo",
    ],
    "Saúde": [
        r"\bsaude", r"\bsaudavel", r"\bfisic",
        r"\bmental", r"\bcorpo", r"\bdoenc",
    ],
    "Paz e equilíbrio": [
        r"\bpaz\b", r"\btranquil", r"\bcalm",
        r"\bequilibr", r"\bharmoni", r"\bseren",
        r"\bplenitude", r"\bbem estar", r"\bcontent",
        r"\balegr", r"\bsorr",
    ],
    "Propósito e sentido": [
        r"\bproposito", r"\bsentido", r"\bessencia",
        r"\bautoconhec", r"\bdirecao",
        r"\bvida significativa", r"\bexist", r"\bvalores?\b",
    ],
    "Realização e conquistas": [
        r"\bobjetiv", r"\bconquist", r"\brealiz",
        r"\bsonh", r"\bsucesso", r"\bmeta", r"\bprogresso",
    ],
    "Espiritualidade e gratidão": [
        r"\bdeus\b", r"\bfe\b", r"\brelig",
        r"\bespirit", r"\bgrat", r"\babenco",
        r"\bmilagre", r"\buniverso",
    ],
    "Presença, lazer e natureza": [
        r"\bmomento", r"\bpresente\b", r"\bdesfrut",
        r"\bnatureza", r"\bpets?\b", r"\bmusica",
        r"\bdanca", r"\bpraia", r"\bviaj",
        r"\blazer", r"\bpequenas? coisas",
    ],
    "Trabalho e estabilidade": [
        r"\btrabalho", r"\bemprego", r"\bdinheiro",
        r"\bfinanceir", r"\bestabilidad",
        r"\bseguranca material",
    ],
    "Liberdade e autonomia": [
        r"\bliberdade", r"\bautonomia", r"\bindepend",
        r"\blivre\b", r"\bescolh",
    ],
    "Aceitação e autocuidado": [
        r"\baceit", r"\bacolh", r"\bautocuidado",
        r"\bcuidar de si", r"\bconsigo mesm",
        r"\bautoaceit", r"\bamor proprio",
    ],
}

concept_matrix = pd.DataFrame(index=base.index)

for concept_name, patterns in concepts.items():
    concept_matrix[concept_name] = (
        base["texto_normalizado"]
        .apply(lambda text: hit(text, patterns))
    )

concept_matrix.head()


,Relações afetivas,Saúde,Paz e equilíbrio,Propósito e sentido,Realização e conquistas,Espiritualidade e gratidão,"Presença, lazer e natureza",Trabalho e estabilidade,Liberdade e autonomia,Aceitação e autocuidado
0,1,0,1,0,1,1,1,0,0,0
1,0,0,0,0,0,1,0,0,0,0
2,1,1,0,0,1,0,0,0,0,0
3,1,0,0,0,0,0,0,0,0,0
4,1,0,1,0,0,0,0,0,0,0


## 4. Construção da matriz categórica

In [ ]:
active = pd.DataFrame(index=base.index)

# Presença ou ausência de cada conceito.
for concept_name in concept_matrix.columns:
    active[f"Conceito:{concept_name}"] = (
        concept_matrix[concept_name]
        .map({1: "Sim", 0: "Não"})
    )

# Escores PERMA discretizados em tercis.
for score in [
    "Escore_P",
    "Escore_E",
    "Escore_R",
    "Escore_M",
    "Escore_A",
    "Escore_Geral",
]:
    if score in base.columns:
        active[f"PERMA:{score}"] = (
            tercile(base[score])
            .fillna("Não informado")
        )

# Faixa etária.
if "Idade" in base.columns:
    active["Sociodemográfico:Faixa etária"] = (
        pd.cut(
            base["Idade"],
            bins=[0, 35, 50, 65, 120],
            labels=["19-35", "36-50", "51-65", "66+"],
            include_lowest=True,
        )
        .astype(object)
        .fillna("Não informado")
    )

# Gênero.
if "Genero" in base.columns:
    active["Sociodemográfico:Gênero"] = (
        base["Genero"]
        .map({1: "Feminino", 2: "Masculino"})
        .fillna("Outro/não informado")
    )

# Escolaridade.
if "Escolaridade" in base.columns:
    active["Sociodemográfico:Escolaridade"] = (
        pd.cut(
            base["Escolaridade"],
            bins=[0, 4, 6, 7, 20],
            labels=[
                "Fundamental/médio",
                "Graduação",
                "Especialização",
                "Mestrado/doutorado",
            ],
            include_lowest=True,
        )
        .astype(object)
        .fillna("Não informado")
    )

# Cluster exploratório, quando existente.
if "Cluster_Exploratorio" in base.columns:
    active["Perfil:Cluster exploratório"] = (
        base["Cluster_Exploratorio"]
        .fillna("Não informado")
        .astype(str)
    )

# Remove variáveis sem variação.
active = active.loc[:, active.nunique() > 1].astype(str)

print(f"Participantes: {active.shape[0]}")
print(f"Variáveis categóricas: {active.shape[1]}")


Participantes: 87
Variáveis categóricas: 20


## 5. Execução da MCA

In [ ]:
# Matriz disjuntiva completa.
G = pd.get_dummies(
    active,
    prefix_sep="=",
).astype(float)

Z = G.to_numpy()
N = Z.sum()
P = Z / N

row_masses = P.sum(axis=1)
column_masses = P.sum(axis=0)

# Matriz padronizada da análise de correspondência.
standardized = (
    np.diag(1 / np.sqrt(row_masses))
    @ (P - np.outer(row_masses, column_masses))
    @ np.diag(1 / np.sqrt(column_masses))
)

# Decomposição em valores singulares.
U, singular_values, VT = np.linalg.svd(
    standardized,
    full_matrices=False,
)

eigenvalues = singular_values ** 2
inertia_percent = 100 * eigenvalues / eigenvalues.sum()

# Coordenadas dos participantes.
row_coordinates = (
    np.diag(1 / np.sqrt(row_masses))
    @ U
    @ np.diag(singular_values)
)

participants = pd.DataFrame({
    "ID": (
        base["ID"].values
        if "ID" in base.columns
        else np.arange(1, len(base) + 1)
    ),
    "dim1": row_coordinates[:, 0],
    "dim2": row_coordinates[:, 1],
})

# Coordenadas das modalidades.
column_coordinates = (
    np.diag(1 / np.sqrt(column_masses))
    @ VT.T
    @ np.diag(singular_values)
)

categories = pd.DataFrame({
    "categoria": G.columns,
    "dim1": column_coordinates[:, 0],
    "dim2": column_coordinates[:, 1],
})

categories["variavel"] = (
    categories["categoria"]
    .str.split("=")
    .str[0]
)

categories["modalidade"] = (
    categories["categoria"]
    .str.split("=")
    .str[1]
)

# Mantém somente a modalidade "Sim" dos conceitos.
concept_points = categories[
    categories["variavel"].str.startswith("Conceito:")
    & categories["modalidade"].eq("Sim")
].copy()

concept_points["label"] = (
    concept_points["variavel"]
    .str.replace("Conceito:", "", regex=False)
    + ": Sim"
)

print(
    f"Dimensão 1: {inertia_percent[0]:.1f}% da inércia\n"
    f"Dimensão 2: {inertia_percent[1]:.1f}% da inércia"
)


Dimensão 1: 16.2% da inércia
Dimensão 2: 8.3% da inércia


## 6. Figura final

In [ ]:
# Cores dos losangos dos conceitos.
concept_colors = {
    "Liberdade e autonomia: Sim": "#27C3D4",
    "Trabalho e estabilidade: Sim": "#C6C51A",
    "Propósito e sentido: Sim": "#9467BD",
    "Presença, lazer e natureza: Sim": "#7F7F7F",
    "Relações afetivas: Sim": "#FF7F0E",
    "Saúde: Sim": "#2CA02C",
    "Paz e equilíbrio: Sim": "#D62728",
    "Realização e conquistas: Sim": "#8C564B",
    "Espiritualidade e gratidão: Sim": "#E377C2",
    "Aceitação e autocuidado: Sim": "#1F77B4",
}

fig, ax = plt.subplots(figsize=(12, 8))

# Pontos dos participantes.
ax.scatter(
    participants["dim1"],
    participants["dim2"],
    s=28,
    color="#5DA5DA",
    alpha=0.58,
    edgecolors="#5DA5DA",
    linewidths=0.4,
    zorder=2,
)

# Linhas de referência dos eixos fatoriais.
ax.axhline(
    0,
    color="#A9C7DD",
    linewidth=1.0,
    alpha=0.8,
    zorder=1,
)
ax.axvline(
    0,
    color="#A9C7DD",
    linewidth=1.0,
    alpha=0.8,
    zorder=1,
)

# Losangos e rótulos dos conceitos.
for _, point in concept_points.iterrows():
    label = point["label"]
    color = concept_colors.get(label, "#333333")

    ax.scatter(
        point["dim1"],
        point["dim2"],
        s=120,
        marker="D",
        color=color,
        edgecolors="white",
        linewidths=0.8,
        zorder=4,
    )

    ax.annotate(
        label,
        xy=(point["dim1"], point["dim2"]),
        xytext=(5, 3),
        textcoords="offset points",
        fontsize=9,
        ha="left",
        va="center",
        zorder=5,
    )

ax.set_title(
    "MCA: participantes projetados no espaço semântico da felicidade",
    fontsize=15,
    pad=12,
)

ax.set_xlabel(
    f"Dimensão 1 ({inertia_percent[0]:.1f}% da inércia)",
    fontsize=11,
)

ax.set_ylabel(
    f"Dimensão 2 ({inertia_percent[1]:.1f}% da inércia)",
    fontsize=11,
)

ax.tick_params(axis="both", labelsize=9)
ax.set_facecolor("#F7F7F7")

for spine in ax.spines.values():
    spine.set_linewidth(0.8)
    spine.set_color("#444444")

ax.margins(x=0.05, y=0.05)
plt.tight_layout()

plt.savefig(
    IMAGE_OUT,
    dpi=300,
    bbox_inches="tight",
)

plt.show()

print(f"Imagem salva em: {IMAGE_OUT}")


### Interpretação da figura

Cada ponto azul representa um participante. Os losangos coloridos representam a modalidade **“Sim”** dos conceitos identificados nas respostas abertas. Participantes e conceitos próximos no mapa apresentam maior associação estrutural no espaço produzido pela MCA.